# Bootstrap Paired Difference Plots

Visualises the paired bootstrap distributions for two model comparisons:
- **PPG − Demographics**: is the PPG model better than a demographics-only baseline?
- **PPG − PPG+Demographics**: what is the cost (or gain) of adding demographics to PPG features?

Loads pre-computed CI artifacts from `Experiment_Bootstrap_CI.py` (point estimates and CI bounds)  
and raw distribution CSVs from `Experiment_Bootstrap.py` (for the histograms).

Metric displayed: **R² (converted to %)**

In [ ]:
import sys
from pathlib import Path

# figures/ → fig_style.py  |  bp_lgbm/ → local_paths.py
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import PERFORMANCE_RESULTS_PAPER, FIGURES_PAPER

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import matplotlib.colors as mcolors

METRIC  = "R2"
SCALE   = 100.0
ALPHA   = 0.05
TARGETS = ["SBP", "DBP", "MAP"]

COMPARISONS = [
    {
        "name":    "ppg_minus_demo",
        "model_a": "ppg",
        "model_b": "demo",
        "title":   "PPG − Demographics",
        "stem":    "Fig_Bootstrap_PPG_minus_Demo",
    },
    {
        "name":    "ppg_minus_ppg_demographics",
        "model_a": "ppg",
        "model_b": "ppg_demo",
        "title":   "PPG − PPG+Demographics",
        "stem":    "Fig_Bootstrap_PPG_minus_PPGDemo",
    },
]

BOOT_ROOT = PERFORMANCE_RESULTS_PAPER / "Bootstrap"
FIG_OUT   = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)

N_BINS = 60

# ── Row colors (one per comparison row) ───────────────────────────────────────
ROW_COLORS = [
    "#2E75B6",   # row 1 — steel blue  (PPG vs Demographics)
    "#D4700A",   # row 2 — amber       (PPG vs PPG+Demographics)
]


def darken(hex_color, factor=0.68):
    """Return a slightly darker version of hex_color (factor < 1 darkens)."""
    r, g, b = mcolors.to_rgb(hex_color)
    return (r * factor, g * factor, b * factor)

In [34]:
# ── Load raw per-model distributions ─────────────────────────────────────────
# Shape: (1000, n_metrics) per model per target
MODEL_KEYS = ["ppg", "demo", "ppg_demo"]

dist = {}  # dist[model_key][target] = pd.Series of METRIC values
for mk in MODEL_KEYS:
    dist[mk] = {}
    for target in TARGETS:
        path = BOOT_ROOT / mk / f"Distribution_{target}.csv"
        df   = pd.read_csv(path)
        dist[mk][target] = df[METRIC].values * SCALE
        print(f"  Loaded {mk}/{target}: {len(dist[mk][target])} resamples")

  Loaded ppg/SBP: 1000 resamples
  Loaded ppg/DBP: 1000 resamples
  Loaded ppg/MAP: 1000 resamples
  Loaded demo/SBP: 1000 resamples
  Loaded demo/DBP: 1000 resamples
  Loaded demo/MAP: 1000 resamples
  Loaded ppg_demo/SBP: 1000 resamples
  Loaded ppg_demo/DBP: 1000 resamples
  Loaded ppg_demo/MAP: 1000 resamples


In [35]:
# ── Compute paired differences ────────────────────────────────────────────────
# Same bootstrap draws → column-wise subtraction is exact pairing
paired_dist = {}  # paired_dist[comp_name][target] = array of differences

for comp in COMPARISONS:
    name = comp["name"]
    paired_dist[name] = {}
    for target in TARGETS:
        paired_dist[name][target] = dist[comp["model_a"]][target] - dist[comp["model_b"]][target]

In [36]:
# ── Load pre-computed CI artifacts ────────────────────────────────────────────
# CI CSVs are semicolon-delimited (sep=";") — distribution CSVs use commas

def load_ci_for_metric(comp_name, target, metric=METRIC, scale=SCALE):
    """Return (point_estimate, ci_lower, ci_upper) scaled by `scale`."""
    path = BOOT_ROOT / "paired" / comp_name / f"CI_{target}.csv"
    ci_df = pd.read_csv(path, sep=";")
    row   = ci_df[ci_df["metric"] == metric].iloc[0]
    return (
        float(row["point_estimate"]) * scale,
        float(row["ci_lower"])       * scale,
        float(row["ci_upper"])       * scale,
    )

# Pre-load everything
ci_data = {}   # ci_data[comp_name][target] = (point_est, lower, upper)
for comp in COMPARISONS:
    name = comp["name"]
    ci_data[name] = {}
    for target in TARGETS:
        ci_data[name][target] = load_ci_for_metric(name, target)
        pt, lo, hi = ci_data[name][target]
        print(f"  {name}/{target}: point={pt:.2f}%  95% CI [{lo:.2f}%, {hi:.2f}%]")

  ppg_minus_demo/SBP: point=30.36%  95% CI [21.75%, 38.86%]
  ppg_minus_demo/DBP: point=30.22%  95% CI [16.70%, 44.01%]
  ppg_minus_demo/MAP: point=36.88%  95% CI [24.72%, 49.03%]
  ppg_minus_ppg_demographics/SBP: point=4.83%  95% CI [0.80%, 9.14%]
  ppg_minus_ppg_demographics/DBP: point=3.68%  95% CI [-1.76%, 10.22%]
  ppg_minus_ppg_demographics/MAP: point=4.83%  95% CI [0.07%, 9.91%]


In [ ]:
def plot_paired_row(comp, axes, color, width_in):
    """Fill one row of axes with the paired-difference distribution per target."""
    name       = comp["name"]
    color_dark = darken(color, factor=0.68)   # point-estimate line
    is_small   = width_in < 5
    label_fs   = 5.5 if is_small else 7.5
    tick_fs    = 6   if is_small else 8
    title_fs   = 7   if is_small else 9
    leg_fs     = 5   if is_small else 7

    for col_idx, (ax, target) in enumerate(zip(axes, TARGETS)):
        values     = paired_dist[name][target]
        pt, lo, hi = ci_data[name][target]

        # Histogram
        ax.hist(values, bins=N_BINS, density=True,
                color=color, alpha=0.45, edgecolor="white", linewidth=0.4)

        # KDE overlay
        kde    = gaussian_kde(values, bw_method="scott")
        x_grid = np.linspace(values.min(), values.max(), 400)
        ax.plot(x_grid, kde(x_grid), color=color, linewidth=1.5)

        # CI shaded region + dashed boundary lines
        ax.axvspan(lo, hi, alpha=0.12, color=color)
        ax.axvline(lo, color=color, linewidth=1.2, linestyle="--")
        ax.axvline(hi, color=color, linewidth=1.2, linestyle="--")

        # Point estimate — same hue as row, darkened
        ax.axvline(pt, color=color_dark, linewidth=1.8, linestyle="-",
                   label=f"Point est. = {pt:.2f}%")

        # Zero reference — dashed black
        ax.axvline(0, color="black", linewidth=1.6, linestyle="--",
                   label="No difference (0%)")

        ax.set_title(target, fontsize=title_fs, fontweight="bold",
                     fontfamily=FONT_FAMILY)
        ax.set_xlabel("ΔR² (%)", fontsize=label_fs, fontfamily=FONT_FAMILY)

        # Leftmost panel carries the row label above "Density"
        if col_idx == 0:
            ax.set_ylabel(
                f"{comp['title']}\n\nDensity",
                fontsize=label_fs, fontfamily=FONT_FAMILY,
            )
        else:
            ax.set_ylabel("Density", fontsize=label_fs, fontfamily=FONT_FAMILY)

        ax.legend(fontsize=leg_fs, framealpha=0.9, edgecolor="#CCCCCC")
        apply_base_style(ax, grid_axis="both")
        ax.tick_params(axis="both", labelsize=tick_fs)

In [ ]:
def make_combined_fig(width_in: float):
    height_in = width_in * ASPECT
    fig, axes = plt.subplots(
        2, 3,
        figsize=(width_in, height_in),
        dpi=DPI,
        layout="constrained",
    )
    for row_idx, (comp, color) in enumerate(zip(COMPARISONS, ROW_COLORS)):
        plot_paired_row(comp, axes[row_idx], color, width_in)
    return fig


fig_full   = make_combined_fig(W_FULL)
fig_single = make_combined_fig(W_SINGLE)

save_fig(fig_full,   "Fig_Bootstrap_Paired_full",   FIG_OUT)
save_fig(fig_single, "Fig_Bootstrap_Paired_single", FIG_OUT)

print(f"Full-page  : {round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px")
print(f"Single-col : {round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px")
plt.show()